# Imports

In [ ]:
import pathlib
import pandas as pd
import numpy as np
import torch

from hypertrees.models import (
    HyperTreeNetAR,
    HyperTreeAR
)

from experiments.utils import (
    load_experiments_specs,
)

# Load Experiment Specifications

In [ ]:
# Load specifications
ablation_run = "A6"
train_type = "global"
dataset = "rossmann"
experiment_specs = load_experiments_specs(
    dataset=dataset,
    train_type=train_type,
)

# Train and Test Data
train_df = experiment_specs["train"]
test_df = experiment_specs["test"]

# Meta Data
meta = experiment_specs["meta"]
features = meta["features"]
fcst_h = meta["fcst_h"]
freq = meta["freq"]

lags = meta["lags"]
max_lag = meta["max_lag"]
n_series = meta["n_series"]
series_ids = meta["series_ids"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 123

# Hyper-Parameters
config = experiment_specs["config"]
loss_fn = config["general"]["loss_function"]

dl_params = config["deep_learning"]
htnetar_params = config["hyper_treenet_params"]
htnetar_params_lgb = {k: v for k, v in htnetar_params.items() if k not in ["num_boost_round", "embedding_dimension", "use_random_projection"]}
htar_params = config["hyper_tree_params"]
htnetar_network_params = {k: v for k, v in htnetar_params.items() if k not in ["num_boost_round", "eta", "linear_tree"]}
htnetar_network_params["rp_embed_dim"] = max_lag
htnetar_network_params["hidden_dim"] = dl_params["hidden_size"]
htnetar_network_params["dropout"] = dl_params["dropout"]
htnetar_network_params["learning_rate"] = dl_params["learning_rate"]

# Ablation:

Remove tsfeatures from the set of features

### HyperTreeNetAR

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

# Initialize
htnet_ar = HyperTreeNetAR(
    p=max_lag,
    freq=freq,
    fcst_h=fcst_h,
    loss_fn=loss_fn,
    device=device
)

# Train
htnet_ar.train(
    lgb_params=htnetar_params_lgb,
    network_params=htnetar_network_params,
    gradient_mode="separate",
    num_iterations=htnetar_params["num_boost_round"],
    train_data=train_df[["series_id", "date", "value"] + features],
    seed=seed,
)

# Forecast
htnet_ar_fcst = htnet_ar.forecast(
    test_data=test_df[["series_id", "date"] + features]
)

# Add actual values
htnet_ar_fcst = pd.merge(
    htnet_ar_fcst,
    test_df[["dataset", "series_id", "date", "value"]],
    on=["series_id", "date"],
    how="inner"
)

### HyperTreeAR

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

# Initialize
ht_ar = HyperTreeAR(
    p=max_lag,
    freq=freq,
    fcst_h=fcst_h,
    loss_fn=loss_fn
)

# Train model
ht_ar.train(
    lgb_params=htar_params,
    num_iterations=htar_params["num_boost_round"],
    train_data=train_df[["series_id", "date", "value"] + features],
    seed=seed,
)

# Forecast
ht_ar_fcst = ht_ar.forecast(
    test_data=test_df[["series_id", "date"] + features]
)

# Add actual values
ht_ar_fcst = pd.merge(
    ht_ar_fcst,
    test_df[["dataset", "series_id", "date", "value"]],
    on=["series_id", "date"],
    how="inner"
)

# Combine forecasts

In [ ]:
fcsts_df = pd.concat(
    [
        htnet_ar_fcst,
        ht_ar_fcst
    ],
    ignore_index=True
)
fcsts_df["ablation"] = ablation_run

# Output

In [ ]:
repo_root = pathlib.Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

result_path = repo_root / "experiments" / "runs" / "results" / "ablation" / dataset.lower()
result_path.mkdir(parents=True, exist_ok=True)

fcsts_df.to_csv(result_path / f"{ablation_run}_fcsts.csv", index=False)